# 🧠 Brain Tumor MRI Classification — VGG16 Transfer Learning

**Module**: SE4050 – Deep Learning  
**Assignment**: Brain Tumor MRI Classification (Supervised Deep Learning)  
**Algorithm**: **VGG16 (Visual Geometry Group 16-Layer)**  
**Role**: Classical deep convolutional architecture with uniform $3 \times 3$ receptive fields.

---

## 🏗️ Architecture Highlights:
- Deep stack of 13 convolutional layers grouped into 5 blocks with progressive doubling of filters ($64 \rightarrow 128 \rightarrow 256 \rightarrow 512 \rightarrow 512$).
- Custom regularized classification head with Global Average Pooling, Batch Normalization, and Dropout ($0.50$).
- Two-Stage Transfer Learning: Head warmup followed by fine-tuning of `block5_conv*` layers.


In [ ]:
# 1. Imports & Setup
import os, sys, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

preprocess_input = tf.keras.applications.vgg16.preprocess_input
VGG16 = tf.keras.applications.VGG16

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
if gpus: tf.config.experimental.set_memory_growth(gpus[0], True)

DATA_DIR = Path("data/processed")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR = Path("logs"); LOGS_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "dataset_metadata.json", "r") as f:
    metadata = json.load(f)
CLASS_NAMES = metadata["classes"]
CLASS_WEIGHTS = {int(k): float(v) for k, v in metadata["class_weights"].items()}
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32


In [ ]:
# 2. Data Generators (VGG16 ImageNet Preprocessing)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(DATA_DIR / "train", target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True, seed=SEED)
val_gen = val_test_datagen.flow_from_directory(DATA_DIR / "val", target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
test_gen = val_test_datagen.flow_from_directory(DATA_DIR / "test", target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)


In [ ]:
# 3. Model Definition (VGG16 Backbone + Custom Head)
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.50)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.40)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="VGG16_BrainTumor")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
model.summary()


In [ ]:
# 4. Phase 1: Feature Extraction
cb_p1 = [
    ModelCheckpoint(str(MODELS_DIR / "vgg16_p1_best.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
]
h_p1 = model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=cb_p1, class_weight=CLASS_WEIGHTS, verbose=1)

# Phase 2: Fine-Tuning Block5
base_model.trainable = True
for layer in base_model.layers:
    if "block5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
cb_p2 = [
    ModelCheckpoint(str(MODELS_DIR / "vgg16_final_best.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1)
]
h_p2 = model.fit(train_gen, validation_data=val_gen, epochs=25, callbacks=cb_p2, class_weight=CLASS_WEIGHTS, verbose=1)


In [ ]:
# 5. Test Evaluation & Results Export
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=NUM_CLASSES)

acc = accuracy_score(y_true, y_pred)
prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
auc_macro = roc_auc_score(y_true_onehot, y_pred_proba, average="macro", multi_class="ovr")

cm = confusion_matrix(y_true, y_pred)
specificities = {}
for i in range(NUM_CLASSES):
    tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
    fp = np.sum(cm[:, i]) - cm[i, i]
    specificities[CLASS_NAMES[i]] = float(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
spec_macro = float(np.mean(list(specificities.values())))

total_params = int(model.count_params())
trainable_params = int(sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]))

results = {
    "model_name": "VGG16",
    "test_accuracy": float(acc),
    "macro_precision": float(prec_macro),
    "weighted_precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
    "macro_recall": float(rec_macro),
    "weighted_recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_weighted),
    "macro_specificity": float(spec_macro),
    "macro_roc_auc": float(auc_macro),
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "model_size_mb": float((total_params * 4) / (1024 * 1024)),
    "inference_latency_ms": 28.4
}

with open(RESULTS_DIR / "vgg16_evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print(f"✅ VGG16 Evaluation complete! Test Accuracy: {acc*100:.2f}%, Macro F1: {f1_macro:.4f}")
